# Analýza cien elektriny na Slovensku

Hodinové day-ahead ceny (ENTSO-E, bez DPH a poplatkov) od roku 2015 z databázy `prices.db`.
Najprv spusti `main.py`, aby databáza existovala a bola aktuálna.

> ⚠️ Pred commitom daj v notebooku **Clear All Outputs** – grafy a tabuľky sú odvodené dáta
> a licencia ich nedovoľuje zverejňovať (repozitár je verejný).

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Chart style: light surface, hairline grid, thin lines
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'figure.facecolor': '#fcfcfb',
    'axes.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7',
    'axes.labelcolor': '#52514e',
    'axes.titlecolor': '#0b0b0b',
    'axes.titlesize': 13,
    'axes.titlelocation': 'left',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'axes.axisbelow': True,
    'grid.color': '#e1e0d9',
    'grid.linewidth': 0.8,
    'xtick.color': '#898781',
    'ytick.color': '#898781',
    'lines.linewidth': 2,
    'legend.frameon': False,
})
colors = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']  # blue, orange, aqua, yellow - always in this order
label_color = '#52514e'
muted = '#a8a7a0'  # gray for background dots
month_names = ['jan', 'feb', 'mar', 'apr', 'máj', 'jún', 'júl', 'aug', 'sep', 'okt', 'nov', 'dec']
weekday_names = ['po', 'ut', 'st', 'št', 'pi', 'so', 'ne']


def label_ends(ax, lines):
    """Write each line's name at its right end; labels that would overlap are pushed apart."""
    ymin, ymax = ax.get_ylim()
    gap = (ymax - ymin) * 0.06
    last_y = None
    for name, x, y in sorted(lines, key=lambda line: line[2]):  # lowest first
        if last_y is not None and y - last_y < gap:
            y = last_y + gap
        ax.annotate(name, xy=(x, y), xytext=(6, 0), textcoords='offset points', color=label_color, va='center')
        last_y = y

## 1. Načítanie dát

`pd.read_sql` načíta výsledok SQL dotazu rovno do tabuľky (DataFrame).
Z `date` + `hour` spravíme jeden časový stĺpec a použijeme ho ako index – pandas potom vie
robiť veci ako „priemer za mesiac“ (`resample`) alebo vybrať obdobie (`df.loc['2024':]`).

In [ ]:
assert os.path.exists('prices.db'), 'prices.db neexistuje - najprv spusti main.py'

conn = sqlite3.connect('prices.db')
df = pd.read_sql('SELECT date, hour, price FROM prices ORDER BY date, hour', conn)
conn.close()

# '2026-10-20' + 13 h -> 2026-10-20 13:00
df['time'] = pd.to_datetime(df['date']) + pd.to_timedelta(df['hour'], unit='h')
df = df.set_index('time')

# helper columns for grouping
df['year'] = df.index.year
df['month'] = df.index.month
df['weekday'] = df.index.dayofweek  # 0 = Monday ... 6 = Sunday
df['day_type'] = np.where(df['weekday'] >= 5, 'víkend', 'pracovný deň')

# today's market (after the 2022 crisis, with lots of solar) - used in many sections below
recent = df.loc['2024-01-01':]
df

## 2. Kontrola dát

Skôr než niečo kreslíme: koľko je dát, aké sú extrémy a či niečo nechýba.

In [ ]:
df['price'].describe().round(3)

In [ ]:
# days that are missing in the data source
all_days = pd.date_range(df.index.min().normalize(), df.index.max().normalize(), freq='D')
missing = all_days.difference(pd.to_datetime(df['date'].unique()))
print(f'Chýbajúce dni ({len(missing)}):', [d.date().isoformat() for d in missing])

print('\nNajdrahšie hodiny:')
print(df['price'].nlargest(5))
print('\nNajlacnejšie hodiny:')
print(df['price'].nsmallest(5))

## 3. Rozdelenie cien

**Histogram**: os x = cena, výška stĺpca = koľko hodín malo takú cenu.

- **Priemer vs. medián** – medián je prostredná hodnota (polovica hodín je lacnejšia, polovica drahšia).
  Ak je priemer vyšší ako medián, ťahajú ho hore občasné veľmi drahé hodiny.
- **Smerodajná odchýlka (std)** – o koľko sa cena typicky líši od priemeru.
- **Šikmosť (skew)** – 0 = symetrické rozdelenie, > 0 = dlhý chvost doprava (drahé výkyvy),
  < 0 = chvost doľava.
- **Percentil** – 95. percentil = cena, pod ktorou je 95 % hodín.

In [ ]:
prices = recent['price']
mean, median = prices.mean(), prices.median()

fig, ax = plt.subplots()
# one bin = 0.01 EUR/kWh; the few extremes outside -0.2 .. 0.6 are left out
ax.hist(prices, bins=np.arange(-0.2, 0.6, 0.01), color=colors[0], edgecolor='#fcfcfb', linewidth=0.5)
ax.axvline(median, color=colors[1], label=f'medián {median:.3f}')
ax.axvline(mean, color=colors[2], label=f'priemer {mean:.3f}')
ax.set_xlabel('€/kWh')
ax.set_title('Rozdelenie hodinových cien od 2024 (počet hodín)')
ax.legend()
plt.show()

for p, v in zip([5, 25, 50, 75, 95], np.percentile(prices, [5, 25, 50, 75, 95])):
    print(f'{p:>2}. percentil: {v:.3f} €/kWh')

# the same statistics for the calm years and for today's market
pd.DataFrame({
    '2015–2020': df.loc[:'2020', 'price'].agg(['mean', 'median', 'std', 'skew']),
    'od 2024': prices.agg(['mean', 'median', 'std', 'skew']),
}).round(3)

## 4. Odľahlé hodnoty (outliers)

**Boxplot** zhrnie rozdelenie do jedného obrázka:

- **krabica** = stredných 50 % hodnôt (od 1. kvartilu Q1 po 3. kvartil Q3), **čiara v nej** = medián
- **fúzy** siahajú najďalej 1,5 × IQR od krabice (IQR = Q3 − Q1)
- **bodky** = odľahlé hodnoty mimo fúzov

Dve bežné pravidlá na odľahlé hodnoty:

- **IQR pravidlo** – odľahlé je všetko pod Q1 − 1,5·IQR alebo nad Q3 + 1,5·IQR (to isté ako fúzy boxplotu)
- **Z-skóre** – koľko smerodajných odchýlok je hodnota od priemeru; |z| > 3 je odľahlé

In [ ]:
fig, ax = plt.subplots()
ax.boxplot([recent.loc[recent['month'] == m, 'price'] for m in range(1, 13)],
           tick_labels=month_names, widths=0.5,
           boxprops={'color': colors[0]}, whiskerprops={'color': colors[0]}, capprops={'color': colors[0]},
           medianprops={'color': colors[1], 'linewidth': 2},
           flierprops={'marker': 'o', 'markersize': 2, 'markerfacecolor': colors[0],
                       'markeredgecolor': 'none', 'alpha': 0.3})
ax.set_title('Hodinové ceny podľa mesiaca od 2024 (€/kWh)')
plt.show()

q1, q3 = prices.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers_iqr = prices[(prices < low) | (prices > high)]
print(f'IQR pravidlo: bežné ceny {low:.3f} až {high:.3f} €/kWh, mimo: {len(outliers_iqr)} hodín ({len(outliers_iqr) / len(prices):.1%})')

z = (prices - prices.mean()) / prices.std()
outliers_z = prices[z.abs() > 3]
print(f'Z-skóre |z| > 3: {len(outliers_z)} hodín ({len(outliers_z) / len(prices):.1%})')

## 5. Vývoj ceny 2015 – 2026

`resample('MS')` zoskupí hodiny po mesiacoch (MS = month start) a `.mean()` z každého mesiaca spraví priemer.

In [ ]:
monthly = df['price'].resample('MS').mean()

fig, ax = plt.subplots()
ax.plot(monthly.index, monthly.values, color=colors[0])
peak = monthly.idxmax()
ax.annotate(f'{peak:%m/%Y}: {monthly.max():.3f} €/kWh', xy=(peak, monthly.max()),
            xytext=(8, -4), textcoords='offset points', color=label_color)
ax.set_title('Priemerná mesačná cena elektriny (€/kWh)')
ax.set_ylim(bottom=0)
plt.show()

df.groupby('year')['price'].agg(['mean', 'min', 'max']).round(3)

## 6. Kĺzavé priemery

**Kĺzavý priemer** (moving average) = priemer posledných N dní, ktorý sa posúva deň po dni.
Vyhladí denný šum a ukáže trend. Čím dlhšie okno, tým hladšia čiara, ale tým neskôr reaguje na zmenu.

- `rolling(30).mean()` = priemer daného dňa a 29 dní pred ním
- `ewm(span=30).mean()` = exponenciálny kĺzavý priemer, novšie dni majú väčšiu váhu, reaguje rýchlejšie

Bodky = denné priemery, čiary = kĺzavé priemery. Chýbajúce dni doplníme `interpolate()`
(hodnota na priamke medzi susednými dňami), inak by kĺzavý priemer mal v tých miestach diery.

In [ ]:
daily_mean = df['price'].resample('D').mean().interpolate()  # one average price per day
ma30 = daily_mean.rolling(30).mean()
ma365 = daily_mean.rolling(365).mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(daily_mean.index, daily_mean.values, s=3, color=muted, label='denný priemer')
ax.plot(ma30.index, ma30.values, color=colors[0], label='kĺzavý priemer 30 dní')
ax.plot(ma365.index, ma365.values, color=colors[1], label='kĺzavý priemer 365 dní')
ax.set_title('Denné ceny a kĺzavé priemery (€/kWh)')
ax.legend(loc='upper left')
plt.show()

**Pásmo bežných hodnôt**: kĺzavý priemer ± 2 kĺzavé smerodajné odchýlky (`rolling(30).std()`).
Keby boli ceny normálne rozdelené, asi 95 % dní by ležalo v pásme. Dni mimo pásma sú na danú dobu nezvyčajné.

In [ ]:
recent_daily = daily_mean.loc['2024-01-01':]
mid = recent_daily.rolling(30).mean()
std = recent_daily.rolling(30).std()
upper, lower = mid + 2 * std, mid - 2 * std
outside = (recent_daily > upper) | (recent_daily < lower)

fig, ax = plt.subplots(figsize=(10, 5), layout='constrained')  # constrained = makes room for the legend
ax.fill_between(mid.index, lower, upper, color=colors[0], alpha=0.12, linewidth=0, label='priemer ± 2 std')
ax.plot(mid.index, mid.values, color=colors[0], label='kĺzavý priemer 30 dní')
ax.scatter(recent_daily.index[~outside], recent_daily[~outside], s=8, color=muted, label='deň')
ax.scatter(recent_daily.index[outside], recent_daily[outside], s=16, color=colors[1],
           label=f'deň mimo pásma ({outside.sum()})')
ax.set_title('Denné ceny od 2024 a pásmo bežných hodnôt (€/kWh)')
ax.legend(loc='upper left', bbox_to_anchor=(0, -0.08), ncols=4)  # legend below the chart
plt.show()

print(f'V pásme: {1 - outside.sum() / mid.notna().sum():.1%} dní')

## 7. Rozklad časového radu

Časový rad sa dá rozložiť na tri časti (aditívny model): **cena = trend + sezónnosť + zvyšok**

- **trend** – dlhodobý smer; tu 7-dňový **centrovaný** kĺzavý priemer (3 dni pred, deň, 3 dni po)
- **sezónnosť** – vzor, ktorý sa pravidelne opakuje; tu týždenný = priemerná odchýlka od trendu pre každý deň v týždni
- **zvyšok** – čo ostane: náhodný šum a nezvyčajné udalosti

Robíme to ručne za posledných 120 dní, aby bolo vidno, ako to funguje
(knižnica `statsmodels` to vie jedným príkazom `seasonal_decompose`).

In [ ]:
last = daily_mean.iloc[-120:]
trend = last.rolling(7, center=True).mean()
detrended = last - trend

effect = detrended.groupby(detrended.index.dayofweek).mean()  # average deviation for Mon ... Sun
effect = effect - effect.mean()                                # the 7 effects add up to 0
seasonal = pd.Series(effect.loc[last.index.dayofweek].values, index=last.index)
residual = last - trend - seasonal

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
for ax, series, name in zip(axes, [last, trend, seasonal, residual],
                            ['cena', 'trend', 'sezónnosť (týždenná)', 'zvyšok']):
    ax.plot(series.index, series.values, color=colors[0], linewidth=1.5)
    ax.set_title(name, fontsize=11)
for ax in axes[2:]:
    ax.axhline(0, color='#c3c2b7', linewidth=1)
fig.suptitle('cena = trend + sezónnosť + zvyšok (posledných 120 dní, €/kWh)', x=0.01, ha='left', fontsize=13)
fig.tight_layout()
plt.show()

effect.index = weekday_names
effect.round(3).rename('vplyv dňa v týždni (€/kWh)')

## 8. Kedy je elektrina počas dňa najlacnejšia?

Úroveň cien sa medzi rokmi veľmi líši, preto porovnáme **tvar** dňa vo vybraných rokoch.
`pivot_table` spraví tabuľku: riadky = hodina, stĺpce = rok, hodnota = priemerná cena.

In [ ]:
years = [2016, 2019, 2023, 2025]
profile = df[df['year'].isin(years)].pivot_table(index='hour', columns='year', values='price', aggfunc='mean')

fig, ax = plt.subplots()
for year, color in zip(years, colors):
    ax.plot(profile.index, profile[year], color=color, label=str(year))
label_ends(ax, [(str(year), 23, profile.loc[23, year]) for year in years])
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel('hodina')
ax.set_title('Priemerná cena podľa hodiny dňa (€/kWh)')
ax.legend(ncols=len(years), loc='upper left')
plt.show()

pd.DataFrame({'najlacnejšia hodina': profile.idxmin(), 'najdrahšia hodina': profile.idxmax()})

## 9. Pracovný deň vs. víkend

Pozeráme len posledné obdobie (od 2024), aby výsledok zodpovedal dnešnému trhu.

In [ ]:
week = recent.pivot_table(index='hour', columns='day_type', values='price', aggfunc='mean')

fig, ax = plt.subplots()
for col, color in zip(['pracovný deň', 'víkend'], colors):
    ax.plot(week.index, week[col], color=color, label=col)
label_ends(ax, [(col, 23, week.loc[23, col]) for col in ['pracovný deň', 'víkend']])
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel('hodina')
ax.set_title('Priemerná cena podľa hodiny od 2024 (€/kWh)')
ax.legend(loc='upper left')
plt.show()

recent.groupby('day_type')['price'].mean().round(3)

## 10. Hodina × mesiac (od 2024)

Heatmapa ukáže naraz denný aj ročný cyklus: čím tmavšie, tým drahšie.

In [ ]:
heat = recent.pivot_table(index='hour', columns='month', values='price', aggfunc='mean')
blues = LinearSegmentedColormap.from_list('blues', ['#cde2fb', '#86b6ef', '#3987e5', '#1c5cab', '#0d366b'])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(heat, aspect='auto', cmap=blues, origin='lower')
ax.set_xticks(range(12), month_names)
ax.set_yticks(range(0, 24, 2))
ax.set_ylabel('hodina')
ax.grid(False)
ax.set_title('Priemerná cena podľa mesiaca a hodiny od 2024 (€/kWh)')
fig.colorbar(im, ax=ax, label='€/kWh')
plt.show()

## 11. Záporné ceny

Keď je v sieti prebytok elektriny (napr. veľa slnka cez víkend), cena môže klesnúť pod nulu.
`df['price'] < 0` dá True/False pre každú hodinu a `sum()` spočíta True ako 1.

In [ ]:
negative = (df['price'] < 0).groupby(df['year']).sum()

fig, ax = plt.subplots()
ax.bar(negative.index, negative.values, color=colors[0], width=0.7)
top = negative.idxmax()
ax.annotate(str(negative.max()), xy=(top, negative.max()), xytext=(0, 4),
            textcoords='offset points', ha='center', color=label_color)
ax.set_xticks(negative.index)
ax.set_title(f'Počet hodín so zápornou cenou za rok ({df.index.max().year} len do {df.index.max():%d.%m.})')
plt.show()

## 12. Ako veľmi kolíše cena počas dňa?

Rozdiel medzi najdrahšou a najlacnejšou hodinou dňa. Veľký rozdiel = oplatí sa presúvať spotrebu
(pračka, nabíjanie auta, batéria) do lacných hodín.

In [ ]:
daily_range = df['price'].resample('D').agg(['min', 'max'])
daily_range['spread'] = daily_range['max'] - daily_range['min']
spread_monthly = daily_range['spread'].resample('MS').mean()

fig, ax = plt.subplots()
ax.plot(spread_monthly.index, spread_monthly.values, color=colors[0])
ax.set_ylim(bottom=0)
ax.set_title('Priemerný denný rozdiel max − min cena, po mesiacoch (€/kWh)')
plt.show()

## 13. Súvisí dnešná cena so včerajšou? (regresia a autokorelácia)

**Bodový graf s regresnou priamkou**: každá bodka = jeden deň, x = priemerná cena predchádzajúceho dňa,
y = priemerná cena v ten deň. Priamka `y = a·x + b` je **lineárna regresia** – `np.polyfit` ju nájde
metódou najmenších štvorcov (súčet štvorcov vzdialeností bodiek od priamky je najmenší).

- **a (sklon)** – o koľko sa v priemere zmení dnešná cena, keď bola včerajšia o 1 €/kWh vyššia
- **R²** (0 až 1) – akú časť kolísania dnešnej ceny vysvetlí včerajšia cena

In [ ]:
x = recent_daily.shift(1)  # yesterday
y = recent_daily           # today
ok = x.notna()
a, b = np.polyfit(x[ok], y[ok], 1)  # straight line y = a*x + b
r2 = np.corrcoef(x[ok], y[ok])[0, 1] ** 2

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x, y, s=10, color=colors[0], alpha=0.5, linewidth=0)
xs = np.linspace(x.min(), x.max(), 100)
ax.plot(xs, a * xs + b, color=colors[1], label=f'y = {a:.2f}·x + {b:.3f}   (R² = {r2:.2f})')
ax.grid(True, axis='both')
ax.set_xlabel('priemerná cena včera (€/kWh)')
ax.set_ylabel('priemerná cena dnes (€/kWh)')
ax.set_title('Denná cena dnes vs. včera (od 2024)')
ax.legend(loc='upper left')
plt.show()

**Autokorelácia (ACF)** = korelácia hodinovej ceny s ňou samou posunutou o 1, 2, … 168 hodín
(`shift(lag, freq='h')`). Vrcholy každých 24 h ukazujú denný cyklus, vrchol pri 168 h týždenný cyklus.

In [ ]:
recent_price = recent['price']
lags = range(1, 169)
acf = [recent_price.corr(recent_price.shift(lag, freq='h')) for lag in lags]

fig, ax = plt.subplots()
ax.vlines(lags, 0, acf, color=colors[0], linewidth=1.5)
ax.axhline(0, color='#c3c2b7', linewidth=1)
ax.set_xticks(range(0, 169, 24))
ax.set_xlabel('posun (hodiny)')
ax.set_title('Autokorelácia hodinovej ceny od 2024')
plt.show()

for lag, name in [(1, 'pred hodinou'), (12, 'pred 12 h'), (24, 'pred dňom'), (168, 'pred týždňom')]:
    print(f'{name:>13}: {acf[lag - 1]:.3f}')

## Zhrnutie

Zhrnutie je zámerne bez čísel – odvodené dáta sa podľa licencie nesmú zverejňovať.
Presné hodnoty vypíšu jednotlivé sekcie po spustení (**Run All**).

**Vývoj a trend**
1. **Kríza 2022 je jasne vidieť.** Do roku 2020 boli ceny nízke a stabilné, v rokoch 2021 – 2022 prudko
   vyskočili a potom klesli, ale zostali niekoľkonásobne vyššie ako pred krízou.
2. **Na jeseň 2026 cena znova rastie** – stúpa 30-dňový kĺzavý priemer aj trend z rozkladu časového radu.

**Rozdelenie**
3. Priemer je vyšší ako medián a šikmosť je kladná: rozdelenie má **chvost doprava**, občas sú hodiny veľmi drahé.
4. **Ceny dnes kolíšu oveľa viac** ako v rokoch 2015 – 2020 (vyššia smerodajná odchýlka).
5. Histogram má **druhý vrchol okolo nuly** – sú to hlavne hodiny okolo obeda, keď vyrábajú solárne elektrárne.

**Odľahlé hodnoty**
6. IQR pravidlo označí za odľahlé viac hodín ako z-skóre. Lacné extrémy sú hlavne na jar a začiatkom leta,
   drahé na jeseň a v zime.
7. Pásmo „30-dňový kĺzavý priemer ± 2 std“ obsahuje väčšinu dní, blízko teoretických 95 %.

**Denný a týždenný cyklus**
8. **Najlacnejšia hodina sa presunula z noci na obed** (solárne elektrárne), najdrahšie je stále večer.
9. **Víkend je lacnejší** ako pracovný deň, najviac na obed. Rozklad časového radu ukazuje ten istý
   týždenný vzor: sobota a hlavne nedeľa sú pod trendom.
10. Najlacnejšie sú jarné obedy, najdrahšie letné a septembrové večery.
11. **Hodín so zápornou cenou** v posledných rokoch výrazne pribudlo.
12. **Rozdiel medzi najdrahšou a najlacnejšou hodinou dňa** je dnes niekoľkonásobne väčší ako pred rokom 2021,
    takže presúvanie spotreby do lacných hodín sa oplatí oveľa viac.

**Závislosť v čase**
13. Dnešná cena silno súvisí so včerajšou – body ležia okolo regresnej priamky a R² ukazuje,
    koľko kolísania včerajšia cena vysvetlí.
14. Autokorelácia má vrcholy každých 24 hodín (denný cyklus) a výraznejší vrchol pri 168 hodinách (týždenný cyklus).